# Pattern Matching, Option, Try & Either



## `match` — the basic shape

A `match` expression takes a value, walks an ordered list of `case` clauses, and returns the body of the first case whose pattern matches. The whole thing is an *expression* — it returns a value, just like `if`.

```scala
val description = day match
  case 1 => "monday"
  case 2 => "tuesday"
  case _ => "later in the week"
```

Three things to notice:

- **Top-down evaluation.** The first matching case wins. Order matters.
- **`_` is the wildcard pattern.** It matches anything and binds nothing. Use it as the default last case.
- **The whole expression has a type.** The compiler computes the common type of all the case bodies.

In [ ]:
val day = 3

val description = day match
  case 1 => "monday"
  case 2 => "tuesday"
  case 3 => "wednesday"
  case 4 => "thursday"
  case 5 => "friday"
  case _ => "weekend"

println(description)  // wednesday

// match returns a value — bind it directly
val isWorkday = day match
  case 1 | 2 | 3 | 4 | 5 => true
  case _                  => false

println(isWorkday)  // true

## Pattern kinds — the seven shapes that cover almost everything

The Scala pattern language is small. Most of what you will write falls into seven categories.

| Pattern kind | Example | Matches |
|---|---|---|
| Literal | `case 0 =>` | exactly that value |
| Wildcard | `case _ =>` | anything; binds nothing |
| Variable | `case x =>` | anything; binds to `x` |
| Typed | `case s: String =>` | values of that type; binds |
| Constructor | `case Tx(id, _, _, _) =>` | matches a case class / enum case; destructures |
| Tuple | `case (a, b) =>` | a 2-tuple; binds both halves |
| Sequence | `case List(a, b, _*) =>` | a list with at least 2 elements |

The combinations of these — plus guards, or-patterns, and `@` bindings, all coming up — give you the full surface.

In [ ]:
def describe(x: Any): String = x match
  case 0                  => "zero literal"
  case "scala"            => "the magic string"
  case n: Int             => s"int $n"
  case s: String          => s"string of length ${s.length}"
  case (a, b)             => s"pair $a / $b"
  case List()             => "empty list"
  case List(x)            => s"one-element list with $x"
  case List(_, _, rest*)  => s"list with 2+ elements"
  case _                  => "something else"

println(describe(0))
println(describe(42))
println(describe("scala"))
println(describe("hello"))
println(describe((1, 2)))
println(describe(List()))
println(describe(List(99)))
println(describe(List(1, 2, 3, 4, 5)))
println(describe(3.14))

## Constructor patterns — destructuring case classes and enums

This is where pattern matching shines for data modelling. A constructor pattern matches a case class or enum case *and* extracts its fields by position. The shape is the constructor name followed by patterns for each field — nested patterns work all the way down.

In [ ]:
case class Address(city: String, country: String)
case class Person(name: String, age: Int, address: Address)

val ganesh = Person("ganesh", 30, Address("bengaluru", "india"))

val summary = ganesh match
  case Person(n, a, Address(c, "india")) => s"$n, $a, from $c (in india)"
  case Person(n, a, Address(c, country)) => s"$n, $a, from $c ($country)"

println(summary)

// You can ignore fields you don't care about with _
ganesh match
  case Person(name, _, _) => println(s"name is $name")

// And bind a whole sub-pattern with @
ganesh match
  case Person(_, _, addr @ Address(c, _)) =>
    println(s"address as a whole: $addr — city: $c")

## Tuple, list, and sequence patterns

Tuples, lists, and other sequences have built-in pattern syntax.

- **Tuple** — `(a, b)` matches any 2-tuple and binds both halves.
- **List** — `List()` matches empty; `List(x)` matches one element; `head :: tail` matches non-empty and splits off the first element.
- **Sequence wildcard** — `List(a, b, rest*)` matches a list with at least 2 elements; `rest` is bound to the remaining `Seq`. In Scala 2 this was `_*`.

In [ ]:
// Tuple destructuring
val pair = ("scala", 3)
val (lang, version) = pair          // pattern in val position
println(s"$lang $version")

// List patterns
def sumList(xs: List[Int]): Int = xs match
  case Nil           => 0
  case head :: tail  => head + sumList(tail)

println(sumList(List(1, 2, 3, 4, 5)))  // 15

// Sequence patterns with varargs
def describeList[A](xs: List[A]): String = xs match
  case Nil                 => "empty"
  case List(x)             => s"single: $x"
  case List(x, y)          => s"pair: $x, $y"
  case List(x, y, rest*)   => s"$x, $y, and ${rest.size} more"

println(describeList(List()))
println(describeList(List("a")))
println(describeList(List("a", "b")))
println(describeList(List(1, 2, 3, 4, 5)))

## Guards, or-patterns, and bindings

Three small features that compose into clean cases.

- **Guards** — `case pattern if condition =>` adds a boolean condition that must also hold.
- **Or-patterns** — `case 0 | 1 | 2 =>` matches any of multiple values in one case. The branch sees the value but cannot bind names that differ between alternatives.
- **`@` binding** — `case x @ pattern =>` binds `x` to the whole matched value while the pattern still destructures.

In [ ]:
def categorize(n: Int): String = n match
  case 0                       => "zero"
  case x if x < 0              => s"negative: $x"
  case x if x < 10             => s"small: $x"
  case x if x % 2 == 0         => s"even and big: $x"
  case _                       => "odd and big"

println(categorize(0))
println(categorize(-5))
println(categorize(7))
println(categorize(100))
println(categorize(101))

// Or-patterns
def isVowel(c: Char): Boolean = c match
  case 'a' | 'e' | 'i' | 'o' | 'u' => true
  case _                            => false

println(isVowel('a'))  // true
println(isVowel('b'))  // false

// @ binding — keep both the whole and the pieces
case class Email(to: String, body: String)

val e = Email("ganesh@example.com", "welcome")
e match
  case msg @ Email(addr, _) if addr.endsWith("example.com") =>
    println(s"internal email: $msg")
  case _ => println("external")

## Pattern matching in vals and in for-comprehensions

Patterns work outside `match` too. Wherever the compiler can statically guarantee the pattern matches, you can use it on the left side of a `val` declaration, or as the binder in a `for` comprehension.

If the pattern *might* fail — say you destructure a `List(a, b)` from a list that might be empty — the assignment will throw `MatchError` at runtime. Reserve val-position patterns for tuples and case classes where the shape is guaranteed.

In [ ]:
// Tuple in val position — always safe
val (a, b, c) = (1, "two", 3.0)
println(s"$a $b $c")

// Case class in val position — safe if the shape is known
case class Point(x: Int, y: Int)
val Point(px, py) = Point(3, 4)
println(s"x=$px y=$py")

// List in val position — RISKY, can throw if shape doesn't match
val List(first, second) = List(10, 20)  // ok
println(s"$first $second")
// val List(first, second) = List(1, 2, 3)  // throws MatchError

// Pattern in for-comprehension — common for iterating over Maps
val ages = Map("alice" -> 30, "bob" -> 25)
for (name, age) <- ages do
  println(s"$name is $age")

## Exhaustiveness checking — the compiler's safety net

When you match on a **sealed** type (a sealed trait, a Scala 3 enum, or any other type the compiler can enumerate), the compiler checks that your `match` covers every case. If you miss one, you get a warning. In a Spark pipeline, that warning is your first line of defence against forgetting a status or a join type.

The check doesn't trigger on open types — matching on `Int`, on `Any`, on an unsealed `String` — because the compiler cannot know what values might appear. Make your domain types sealed (case classes inside an enum, or extending a sealed trait) to get this benefit.

In [ ]:
enum Status:
  case Approved, Declined, Pending

import Status.*

// This is exhaustive — compiler is happy
def labelComplete(s: Status): String = s match
  case Approved => "ok"
  case Declined => "no"
  case Pending  => "wait"

// Comment one case out to see the warning at compile time:
//   match may not be exhaustive. It would fail on pattern case: Pending
// def labelIncomplete(s: Status): String = s match
//   case Approved => "ok"
//   case Declined => "no"

println(labelComplete(Pending))

## Partial functions — `{ case ... => ... }` as a value

A *partial function* is a function that may not be defined for all inputs. Scala lets you write one inline with the `{ case ... => ... }` syntax — exactly the shape of a `match` expression's case list, but standalone.

A `PartialFunction[A, B]` has two methods of note:

- **`apply(a)`** — call it; throws `MatchError` if no case matches.
- **`isDefinedAt(a)`** — check whether any case would match, *without* calling.

This is what the `collect` collection method (notebook 04) takes. It is also widely used in actor libraries and in any "dispatch on shape" code.

In [ ]:
val divideBy: PartialFunction[Int, Int] =
  case n if n != 0 => 100 / n

println(divideBy.isDefinedAt(5))   // true
println(divideBy.isDefinedAt(0))   // false
println(divideBy(5))               // 20
// println(divideBy(0))            // throws MatchError

// Used with collect — only defined inputs produce outputs
val xs = List(0, 1, 2, 3, 0, 5)
println(xs.collect(divideBy))      // List(100, 50, 33, 20) — zeros dropped

// Combining partial functions with orElse
val handleEven: PartialFunction[Int, String] =
  case n if n % 2 == 0 => s"even: $n"

val handleOdd: PartialFunction[Int, String] =
  case n if n % 2 == 1 => s"odd: $n"

val handleAny = handleEven.orElse(handleOdd)
println(handleAny(4))   // even: 4
println(handleAny(7))   // odd: 7

## `Option[A]` — encoding "value might be missing"

`Option[A]` is a sealed ADT with exactly two cases:

- `Some(value: A)` — a value is present.
- `None` — no value.

It is the Scala replacement for `null`. The difference: `null` lies about types — a `String` that is sometimes `null` claims to be a `String` always, and you find out the truth at runtime via `NullPointerException`. `Option[String]` tells the truth in the type — every caller is forced to acknowledge the missing case.

The standard library returns `Option` from any method that might legitimately have no answer — `Map.get`, `List.headOption`, `String.toIntOption`. Most java APIs return `null`; wrap them with `Option(javaCall())` at the seam to convert.

In [ ]:
// Construction
val present: Option[Int] = Some(42)
val absent:  Option[Int] = None

println(present)  // Some(42)
println(absent)   // None

// From Map.get
val ages = Map("alice" -> 30, "bob" -> 25)
println(ages.get("alice"))   // Some(30)
println(ages.get("carol"))   // None

// Wrap a possibly-null java call
val raw: String = null
val safe: Option[String] = Option(raw)
println(safe)                // None

val raw2: String = "hello"
println(Option(raw2))        // Some(hello)

// String to Int with safety
println("42".toIntOption)    // Some(42)
println("xyz".toIntOption)   // None

## Option combinators — `map`, `flatMap`, `getOrElse`, `fold`, `filter`

The whole point of `Option` is that you compose operations *without* unwrapping. The combinators automatically propagate `None` through a chain — if any step is empty, the whole result is empty.

- **`map(f)`** — apply `f` to the inner value if present; stay `None` if absent.
- **`flatMap(f)`** — like `map`, but `f` itself returns an `Option`. Avoids `Option[Option[A]]` nesting.
- **`getOrElse(default)`** — extract the value, or use the default if absent.
- **`fold(default)(f)`** — combine `getOrElse` and `map` — provide a default for `None` and a function for `Some`.
- **`filter(p)`** — keep the value if it passes the predicate; otherwise become `None`.

In [ ]:
val nameOpt: Option[String] = Some("ganesh")

// map — transform inside the Option
println(nameOpt.map(_.toUpperCase))   // Some(GANESH)
println(None.map((s: String) => s.toUpperCase))  // None

// flatMap — when the function itself returns an Option
def lookupId(name: String): Option[Int] =
  if name == "ganesh" then Some(42) else None

println(nameOpt.flatMap(lookupId))    // Some(42)
println(Some("alice").flatMap(lookupId))  // None

// getOrElse — extract with a default
println(nameOpt.getOrElse("unknown"))  // ganesh
println(None.getOrElse("unknown"))      // unknown

// fold — handle both branches in one expression
val message = nameOpt.fold("nobody home")(n => s"hello, $n")
println(message)

// filter — drop the value if it fails the predicate
println(Some(42).filter(_ > 10))   // Some(42)
println(Some(5).filter(_ > 10))    // None

## `Try[A]` — encoding "computation might have failed"

`Try[A]` is a sealed ADT for the result of a computation that might throw:

- `Success(value: A)` — completed normally, here is the result.
- `Failure(throwable: Throwable)` — threw, here is the exception.

`Try { expression }` wraps any block — if the block throws, you get `Failure(thrown)`; otherwise `Success(returned)`. This converts exception-based code into value-based code that composes the same way `Option` does.

When to prefer `Try` over plain `try`/`catch`: when you want to *return* the failure as data, chain computations together, or store/log the exception without immediately handling it.

In [ ]:
import scala.util.{Try, Success, Failure}

def parseInt(s: String): Try[Int] = Try(s.toInt)

println(parseInt("42"))      // Success(42)
println(parseInt("abc"))     // Failure(java.lang.NumberFormatException: For input string: "abc")

// Combinators mirror Option's
val result = parseInt("10").map(_ * 2)
println(result)              // Success(20)

val resultFail = parseInt("abc").map(_ * 2)
println(resultFail)          // Failure(...)

// recover — convert a Failure into a Success with a fallback
val safe = parseInt("xyz").recover { case _: NumberFormatException => -1 }
println(safe)                // Success(-1)

// recoverWith — like recover, but the handler returns a Try
val safe2 = parseInt("xyz").recoverWith { case _: NumberFormatException => parseInt("0") }
println(safe2)               // Success(0)

// toOption — drop the exception details
println(parseInt("42").toOption)   // Some(42)
println(parseInt("xyz").toOption)  // None

## `Either[E, A]` — encoding "result is one of two possible types"

`Either[E, A]` has two cases:

- `Left(e: E)` — by convention, the *error* or alternative case.
- `Right(a: A)` — by convention, the *success* or main case. (Mnemonic: *right* sounds like *correct*.)

`Either` is what you reach for when you want typed errors — not exceptions, not just "missing," but a specific error value carrying information about *what* went wrong. The error type `E` is on the left; the success type `A` is on the right.

Scala 3 `Either` is *right-biased* — its `map`, `flatMap`, and `for` combinators operate on the `Right` side. A `Left` short-circuits through the chain, carrying the error untouched.

In [ ]:
sealed trait ParseError
case object EmptyInput     extends ParseError
case class  BadFormat(s: String) extends ParseError
case class  OutOfRange(n: Int)   extends ParseError

def parsePositive(s: String): Either[ParseError, Int] =
  if s.isEmpty then Left(EmptyInput)
  else s.toIntOption match
    case None             => Left(BadFormat(s))
    case Some(n) if n < 0 => Left(OutOfRange(n))
    case Some(n)          => Right(n)

println(parsePositive("42"))   // Right(42)
println(parsePositive(""))     // Left(EmptyInput)
println(parsePositive("xyz"))  // Left(BadFormat(xyz))
println(parsePositive("-3"))   // Left(OutOfRange(-3))

// Right-biased combinators
val doubled = parsePositive("21").map(_ * 2)
println(doubled)              // Right(42)

val doubledFail = parsePositive("xyz").map(_ * 2)
println(doubledFail)          // Left(BadFormat(xyz)) — error carried through

// fold — handle both sides
val message = parsePositive("xyz").fold(
  err => s"failed: $err",
  n   => s"parsed: $n",
)
println(message)

## `for` comprehensions over Option, Try, and Either

`for` comprehensions are not just for collections. Any type with `map`, `flatMap`, and (for guards) `withFilter` can be used in a `for`. `Option`, `Try`, and `Either` all qualify — and `for` becomes the cleanest way to chain operations that may short-circuit.

The pattern: a sequence of `name <- computation` lines, each pulling the success value out of the wrapper. The first failure short-circuits — the rest of the chain is skipped, and the whole expression evaluates to that failure.

In [ ]:
// Option chain — short-circuits on the first None
case class User(id: Int, name: String, managerId: Option[Int])

val users = Map(
  1 -> User(1, "ganesh",  Some(2)),
  2 -> User(2, "alice",   Some(3)),
  3 -> User(3, "bob",     None),
)

def managerName(userId: Int): Option[String] =
  for
    user    <- users.get(userId)
    mgrId   <- user.managerId
    manager <- users.get(mgrId)
  yield manager.name

println(managerName(1))   // Some(alice)
println(managerName(2))   // Some(bob)
println(managerName(3))   // None — bob has no manager
println(managerName(99))  // None — user does not exist

// Either chain — first Left wins
def step1(x: Int): Either[String, Int] = if x > 0 then Right(x * 2) else Left("step1: not positive")
def step2(x: Int): Either[String, Int] = if x < 100 then Right(x + 1) else Left("step2: too large")
def step3(x: Int): Either[String, Int] = Right(x * x)

val result =
  for
    a <- step1(5)
    b <- step2(a)
    c <- step3(b)
  yield c

println(result)              // Right(121)
println(for
  a <- step1(-1)
  b <- step2(a)
  c <- step3(b)
yield c)                     // Left(step1: not positive)

## When to reach for each

A small decision table for the three:

| Situation | Reach for |
|---|---|
| A value might be present or absent, the absent case is normal | `Option` |
| An operation might throw, and you want to handle or report the exception | `Try` |
| An operation might fail with one of several known errors, and you want to carry information about which one | `Either[Error, Result]` |
| You're modeling business logic where "what went wrong" is part of the domain | `Either[YourErrorADT, Result]` |
| You're at the boundary of a java api or unrecoverable exception | `Try` to wrap, then convert to `Either` if appropriate |

In modern Scala codebases — especially anything built on Typelevel libraries — `Either` (or its richer cousins like `EitherT`, `Validated`) is the workhorse. `Option` covers the simpler "present-or-not" case. `Try` shows up at integration boundaries with exception-throwing apis.